# 5a – Reducing embedding dimensions: PCA and UMAP

This is the first of three companion notebooks to Example 5. RAPTOR's clustering step takes one paragraph in the paper and rests on three ideas:

| Notebook | Idea | The question it answers |
|---|---|---|
| **5a** (this one) | Dimensionality reduction: UMAP in the paper, PCA in notebook 5 | Why can't we cluster the 768-dimensional embeddings directly, and what is lost when we squeeze them into a few dimensions? |
| 5b | Gaussian Mixture Models (GMM) | What does it mean to cluster *softly*, and how is a mixture fitted? |
| 5c | Bayesian Information Criterion (BIC) | How many clusters should there be? |

The paper (Section 3) motivates the first idea like this:

> The high dimensionality of vector embeddings presents a challenge for traditional GMMs, as distance metrics may behave poorly when used to measure similarity in high-dimensional spaces (Aggarwal et al., 2001). To mitigate this, we employ Uniform Manifold Approximation and Projection (UMAP), a manifold learning technique for dimensionality reduction (McInnes et al., 2018). The number of nearest neighbors parameter, n_neighbors, in UMAP determines the balance between the preservation of local and global structures.

We check both reasons for reducing on our own data – how distances behave, and how many numbers a Gaussian needs – and then look at the two methods: **PCA**, the best *linear* projection, and **UMAP**, the *non-linear* method of the paper. We visualise what each one keeps and what it loses, and finally measure which of the two keeps the neighbourhoods that clustering depends on, at the 10 dimensions RAPTOR actually clusters in.

No LLM is involved; the notebook runs in well under a minute. The first UMAP call in a fresh environment takes a few extra seconds while `numba` compiles UMAP's code.

## Imports

- `numpy`, `matplotlib.pyplot` – arrays and plots.
- `umap` (package `umap-learn`) – the reference implementation of UMAP by its authors. Besides the `umap.UMAP` class we import two of its building blocks, `smooth_knn_dist` and `find_ab_params`, to look inside the algorithm.
- `sklearn.decomposition.PCA` – principal component analysis.
- `sklearn.manifold.trustworthiness` – a score for how well a low-dimensional map preserves neighbourhoods; it is explained where we first use it.
- `sklearn.datasets.load_digits` – a small classic dataset that ships with scikit-learn, used for one textbook comparison.
- LangChain's splitters and `OllamaEmbeddings` – to chunk and embed our documents exactly as in the earlier notebooks.
- `Counter` – to count chunks per group.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import umap
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
from umap.umap_ import find_ab_params, smooth_knn_dist

## Configuration

- `DATA_DIR`, `EMBEDDING_MODEL` – our four documents, and the Ollama embedding model of Examples 1–3 and 5 (`embeddinggemma:300m`: 768 dimensions, unit-length vectors).
- `CHUNK_SIZE = 400` – for the pictures we want more points than notebook 5's 30 leaves, so we cut smaller chunks: about 90 over the four documents.
- `LEAF_CHUNK_SIZE = 1200` – notebook 5's chunk size. The last section repeats the decisive measurement on exactly its 30 leaves.
- `MIN_CHARS = 80` – fragments shorter than this (date lines, boilerplate) are dropped, as in notebook 5.
- `SEED = 0` – UMAP is a randomised algorithm, and so are the random vectors we compare against; a fixed seed makes every run identical. UMAP runs single-threaded when it gets a seed; passing `n_jobs=1` says so explicitly and avoids a warning.
- `plt.rcParams` – a slightly higher default resolution and smaller titles for all figures.

In [ ]:
DATA_DIR = Path("data")
EMBEDDING_MODEL = "embeddinggemma:300m"

CHUNK_SIZE = 400
LEAF_CHUNK_SIZE = 1200
MIN_CHARS = 80
SEED = 0

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10})

## The data: every chunk of our four documents

Two loaders produce `(text, group)` pairs:

- `load_statements(chunk_size)` chunks `document_a.md` exactly like notebook 5: it undoes the escaped dollar signs, drops the `---` rules, splits at the headings with `MarkdownHeaderTextSplitter`, cuts long sections at paragraph, line, table-row and sentence boundaries, drops short fragments and prepends every chunk's heading path. The *group* is derived from the headings by `part_of_statements`: the four statements, Note 1 (accounting policies), Notes 2–6 (fixed assets, credit lines, debt, leases) and Notes 7–10 (other disclosures).
- `load_text_file(name, group)` splits one of the three short documents of Examples 1–4 with the usual recursive splitter.

All texts are embedded in one call. The groups are **never** shown to PCA or UMAP. We only use them to colour the plots, so that we can judge whether a projection keeps related chunks together.

In [ ]:
def part_of_statements(metadata):
    if metadata.get("h2") != "Notes to Financial Statements":
        return "Employer Group: statements"
    note = int(metadata["h3"].split()[1])
    if note == 1:
        return "Employer Group: Note 1, accounting policies"
    if note <= 6:
        return "Employer Group: Notes 2-6, assets, debt, leases"
    return "Employer Group: Notes 7-10, other"


def load_statements(chunk_size):
    text = (DATA_DIR / "document_a.md").read_text(encoding="utf-8").replace("\\$", "$")
    text = "\n".join(line for line in text.splitlines() if line.strip() != "---")
    headers = [("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4")]
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=0,
        separators=["\n\n", "\n", r"(?<=</tr>)", r"(?<=\. )", " ", ""],
        is_separator_regex=True,
    )
    chunks = []
    for piece in splitter.split_documents(MarkdownHeaderTextSplitter(headers_to_split_on=headers).split_text(text)):
        if len(piece.page_content) < MIN_CHARS:
            continue
        path = " > ".join(piece.metadata[k] for k in ("h1", "h2", "h3", "h4") if k in piece.metadata)
        chunks.append((f"{path}\n\n{piece.page_content}", part_of_statements(piece.metadata)))
    return chunks


def load_text_file(name, group):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=0,
        separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
        is_separator_regex=True,
    )
    text = (DATA_DIR / name).read_text(encoding="utf-8")
    return [(chunk, group) for chunk in splitter.split_text(text) if len(chunk) >= MIN_CHARS]


chunks = (load_statements(CHUNK_SIZE)
          + load_text_file("novaterra_annual_report.txt", "Novaterra: utility annual report")
          + load_text_file("meridian_credit_policy.txt", "Meridian: bank credit policy")
          + load_text_file("harbor_city_transit.txt", "Harbor City: transit review"))
texts = [text for text, _ in chunks]
labels = [group for _, group in chunks]

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
vectors = np.array(embeddings.embed_documents(texts))
print(f"{len(texts)} chunks, vectors of shape {vectors.shape}\n")
for group, count in sorted(Counter(labels).items()):
    print(f"  {count:3d}  {group}")

Two small plotting helpers, used by every map below: `plot_groups(ax, points, title)` draws 2-D points coloured by group, and `legend_below(fig, ax)` puts a single legend under the whole figure instead of covering the points.

In [ ]:
GROUPS = sorted(set(labels))
GROUP_COLORS = {group: plt.get_cmap("tab10")(i) for i, group in enumerate(GROUPS)}


def plot_groups(ax, points, title):
    for group in GROUPS:
        mask = np.array([label == group for label in labels])
        ax.scatter(points[mask, 0], points[mask, 1], s=30, color=GROUP_COLORS[group],
                   edgecolor="white", linewidth=0.5, label=group)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def legend_below(fig, ax):
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=4, fontsize=8, frameon=False)
    fig.tight_layout(rect=(0, 0.09, 1, 1))

## Why reduce? (1) "Near" and "far" look alike in high dimensions

Clustering groups points that are close to each other, which only works if *close* and *far* are clearly different. For random data in high dimensions they are not: the nearest and the farthest neighbour of a point end up at almost the same distance. This is the "surprising behaviour of distance metrics in high dimensional space" (Aggarwal et al., 2001) that the paper cites.

We measure it with one number per point – the distance to its nearest neighbour divided by the distance to its farthest neighbour – and take the median over all points. 0 means a sharp contrast between near and far; 1 means everything is equally far away. `contrast(unit_vectors)` computes it with cosine distance, which for unit vectors is 1 minus the dot product. We compare our embeddings with random unit vectors of increasing dimension, always with the same number of points, so that the comparison is fair.

In [ ]:
def contrast(unit_vectors):
    distances = 1 - unit_vectors @ unit_vectors.T
    np.fill_diagonal(distances, np.nan)
    return float(np.median(np.nanmin(distances, axis=1) / np.nanmax(distances, axis=1)))


rng = np.random.default_rng(SEED)
dimensions = [2, 3, 5, 8, 10, 15, 20, 30, 50, 100, 200, 400, 768]
random_contrast = []
for dimension in dimensions:
    random_vectors = rng.normal(size=(len(vectors), dimension))
    random_contrast.append(contrast(random_vectors / np.linalg.norm(random_vectors, axis=1, keepdims=True)))
our_contrast = contrast(vectors)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.semilogx(dimensions, random_contrast, "o-", label=f"{len(vectors)} random unit vectors")
ax.axhline(our_contrast, color="crimson", ls="--",
           label=f"our {len(vectors)} chunk embeddings, 768-d ({our_contrast:.2f})")
ax.set_xlabel("dimension")
ax.set_ylabel("median of nearest / farthest distance")
ax.set_ylim(0, 1)
ax.set_title("How different are 'near' and 'far'?")
ax.legend()
plt.show()

What the picture says:

- For random vectors the contrast collapses as the dimension grows: among 768-dimensional random vectors, a point's nearest neighbour is typically more than 80% as far away as its farthest one. Neighbourhoods stop meaning anything.
- Our 768-dimensional embeddings behave very differently. Their median ratio is about 0.3, the value random vectors already reach at around 20 dimensions. The embeddings are not spread over all 768 directions; they lie close to a much lower-dimensional surface – a *manifold* – because text has structure: chunks about debt share vocabulary and phrasing, chunks about trams share other vocabulary.

So the concern the paper cites is real for random data but moderate for real embeddings. And their low *intrinsic* dimension is good news: it is the reason a projection to around 10 dimensions can keep most of the neighbourhood structure. The decisive reason to reduce is the next one.

## Why reduce? (2) A Gaussian in 768 dimensions has too many numbers

A GMM (notebook 5b) describes every cluster by a Gaussian: a mean vector with *d* numbers, and a covariance matrix – a symmetric *d* × *d* table that describes the cluster's shape – with *d*(*d* + 1)/2 free numbers. All of them have to be estimated from the points in that cluster.

In [ ]:
for d in [2, 8, 10, 768]:
    covariance = d * (d + 1) // 2
    print(f"d = {d:3d}:  {d:3d} for the mean + {covariance:7,d} for the covariance = {d + covariance:7,d} numbers per cluster")

With 768 dimensions one cluster needs 296,064 numbers, estimated from a few dozen chunks. That is hopeless: the estimated covariance matrix would be singular. Even 8 dimensions turn out to be too many for clusters of three to six leaves – notebook 5c shows exactly what goes wrong then. So the dimension has to come down before a mixture can be fitted: to 10 in the paper, to 8 in notebook 5. The question is how to do that without destroying the structure we want to cluster.

## PCA: the best linear projection

Principal Component Analysis rotates the coordinate system so that the first axis (PC1) points in the direction in which the data varies most, the second axis (PC2) in the direction of largest remaining variance perpendicular to the first, and so on. Keeping the first *m* axes and dropping the rest is the best possible *linear* projection to *m* dimensions: it preserves as much of the variance – the spread of the points around their mean – as any linear projection can.

Properties that matter for us: PCA is deterministic (same input, same output), has no parameters besides *m*, is a single matrix multiplication once fitted, and its *explained variance* tells you exactly how much spread each axis keeps.

The toy example shows the idea in two dimensions, with 150 points from a stretched, tilted cloud. `PCA(2).fit(cloud)` finds the two principal axes: `components_` holds their directions, `explained_variance_` the variance along each (left; each arrow is two standard deviations long). Projecting onto PC1 alone (right) moves every point perpendicularly onto the long axis; the grey lines are what is thrown away.

In [ ]:
rng = np.random.default_rng(SEED)
cloud = rng.multivariate_normal([0, 0], [[3.0, 1.6], [1.6, 1.2]], size=150)
toy_pca = PCA(2).fit(cloud)
projected = toy_pca.mean_ + np.outer(toy_pca.transform(cloud)[:, 0], toy_pca.components_[0])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(cloud[:, 0], cloud[:, 1], s=10, alpha=0.6)
for direction, variance in zip(toy_pca.components_, toy_pca.explained_variance_):
    axes[0].annotate("", xy=toy_pca.mean_ + 2 * np.sqrt(variance) * direction, xytext=toy_pca.mean_,
                     arrowprops={"arrowstyle": "->", "lw": 2.5, "color": "crimson"})
axes[0].set_title("the two principal axes, each two standard deviations long")
for original, image in zip(cloud[::3], projected[::3]):
    axes[1].plot([original[0], image[0]], [original[1], image[1]], color="gray", lw=0.5)
axes[1].scatter(cloud[:, 0], cloud[:, 1], s=10, alpha=0.3)
axes[1].scatter(projected[:, 0], projected[:, 1], s=10, color="crimson")
axes[1].set_title(f"projecting onto PC1 keeps {toy_pca.explained_variance_ratio_[0]:.0%} of the variance")
for ax in axes:
    ax.set_aspect("equal")
plt.show()

### PCA on our embeddings

`PCA().fit(vectors)` computes all components; `explained_variance_ratio_` is each component's share of the total variance.

In [ ]:
pca = PCA().fit(vectors)
cumulative = np.cumsum(pca.explained_variance_ratio_)
print(f"{len(cumulative)} components; {len(vectors)} points span at most {len(vectors) - 1} directions")
for m in [2, 8, 10, 20]:
    print(f"  the first {m:2d} components keep {cumulative[m - 1]:.0%} of the variance")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(range(1, 31), pca.explained_variance_ratio_[:30])
axes[0].set_xlabel("component")
axes[0].set_title("share of the variance per component (first 30)")
axes[1].plot(range(1, len(cumulative) + 1), cumulative)
axes[1].axhline(0.8, color="gray", ls="--")
axes[1].set_xlabel("number of components kept")
axes[1].set_title("cumulative share")
plt.show()

Text embeddings spread their variance over many directions: the first component carries only about a tenth of it, two components less than a fifth, and it takes about 30 components to reach 80%. Note also that *n* points span at most *n* − 1 directions, so only about 90 components carry any variance here, not 768 (for notebook 5's 30 leaves: 29). A 2-D PCA picture therefore shows only a small part of the structure. Here it is:

In [ ]:
pca_2d = PCA(2, random_state=SEED).fit_transform(vectors)

fig, ax = plt.subplots(figsize=(7.5, 6.3))
plot_groups(ax, pca_2d, "PCA: the two directions of largest variance")
legend_below(fig, ax)
plt.show()

The three short documents of Examples 1–4 separate from the financial statements, and the four statements (tables full of figures) sit in a corner of their own. Inside the notes, however, Note 1 and Notes 2–6 overlap heavily: whatever distinguishes them lives in the many directions this picture leaves out.

## UMAP: keep the neighbours, not the variance

PCA asks: *in which directions does the data spread?* UMAP asks a different question: *who is next to whom?* – and tries to draw a map in which the same points are next to each other. It works in four steps:

1. **Neighbour graph.** For every point find its `n_neighbors` nearest neighbours in the original space, with the distance of your choice (the paper's code uses cosine distance, and so do we).
2. **Local scaling.** Turn distances into edge weights between 0 and 1, measured *relative to each point's own neighbourhood*. With $\rho_i$ the distance from point *i* to its nearest neighbour, the edge to neighbour *j* gets the weight $\exp\!\big(-(d(i,j)-\rho_i)/\sigma_i\big)$, where $\sigma_i$ is chosen so that the weights of *i*'s neighbours add up to $\log_2(\texttt{n\_neighbors})$. The nearest neighbour always gets weight 1, however far away it is. This is the "uniform" in UMAP: it assumes the data is spread evenly over the manifold, and stretches or shrinks distances locally until that is true. Dense and sparse regions are treated alike.
3. **Symmetrise.** Point *i* may count *j* among its neighbours but not the other way round. The two directed weights *a* and *b* are combined into *a* + *b* − *ab*, the probability that at least one of two independent events happens. The result is one undirected, weighted graph.
4. **Layout.** Place the points in the target dimension (starting from a spectral layout of the graph) and move them step by step: every edge pulls its endpoints together, and randomly drawn pairs of non-neighbours are pushed apart. The similarity the layout aims for at distance *x* is $1/(1+a\,x^{2b})$, with *a* and *b* derived from the parameter `min_dist`.

Steps 1–3 describe the data; step 4 draws it. The next two figures look at steps 2 and 3 on our chunks.

### Step 2 up close: a dense and an isolated chunk

We compute the cosine distances between all chunks (clipped at 0 against rounding noise), take every chunk's 10 nearest neighbours – the first one is the chunk itself at distance 0, which is UMAP's convention – and hand their distances to UMAP's own `smooth_knn_dist`, which returns $\sigma_i$ and $\rho_i$ for every point. Then we pick two chunks: the one whose 9th neighbour is closest (it sits in the densest region) and the one whose nearest neighbour is farthest away (the most isolated chunk). The curves show the weight each of them gives a neighbour at a given distance; the dots are their actual 9 neighbours.

In [ ]:
K = 10
distances = np.clip(1 - vectors @ vectors.T, 0, None)
np.fill_diagonal(distances, 0)
knn_distances = np.sort(distances, axis=1)[:, :K]
sigmas, rhos = smooth_knn_dist(knn_distances.astype(np.float32), float(K))
dense, isolated = int(np.argmin(knn_distances[:, -1])), int(np.argmax(rhos))

fig, ax = plt.subplots(figsize=(8.5, 4.3))
grid = np.linspace(0, 0.8, 300)
for i, color in [(dense, "tab:blue"), (isolated, "tab:red")]:
    neighbours = knn_distances[i, 1:]
    ax.plot(grid, np.exp(-np.maximum(grid - rhos[i], 0) / sigmas[i]), color=color,
            label=f"chunk {i}: {texts[i].splitlines()[0][:55]}")
    ax.scatter(neighbours, np.exp(-np.maximum(neighbours - rhos[i], 0) / sigmas[i]), color=color)
ax.set_xlabel("cosine distance to a neighbour")
ax.set_ylabel("edge weight")
ax.set_title(f"local scaling of the {K - 1} nearest neighbours of a dense and an isolated chunk")
ax.legend(fontsize=8)
plt.show()
for i in (dense, isolated):
    print(f"chunk {i}: rho = {rhos[i]:.3f}, sigma = {sigmas[i]:.3f}")

The dense chunk's neighbours are all close, and their weights fall off gently. The isolated chunk's nearest neighbour is far away – farther than any neighbour of the dense chunk – yet it still gets weight 1, and the weights of the next neighbours drop steeply right behind it. In UMAP's eyes both chunks have a proper neighbourhood. That is why UMAP pulls isolated points towards the nearest group instead of leaving them far out, and why distances in a UMAP map do not mean what they mean in the original space.

### Steps 3 and 4: the graph and the map

`umap.UMAP(n_neighbors=10, metric="cosine").fit(vectors)` runs all four steps. Afterwards `reducer.graph_` holds the symmetric weighted graph of steps 1–3 as a sparse matrix and `reducer.embedding_` the 2-D layout of step 4. We draw every edge with an opacity proportional to its weight, behind the points.

In [ ]:
reducer = umap.UMAP(n_neighbors=10, metric="cosine", random_state=SEED, n_jobs=1).fit(vectors)
umap_2d = reducer.embedding_
graph = reducer.graph_.tocoo()

fig, ax = plt.subplots(figsize=(8, 6.8))
for i, j, weight in zip(graph.row, graph.col, graph.data):
    if i < j:
        ax.plot(*zip(umap_2d[i], umap_2d[j]), color="black", lw=0.8, alpha=0.6 * float(weight), zorder=0)
plot_groups(ax, umap_2d, f"UMAP layout and its neighbour graph ({graph.nnz // 2} edges, opacity = weight)")
legend_below(fig, ax)
plt.show()

Most strong edges connect chunks of the same colour, and the layout keeps exactly those edges short: the map is a drawing of the graph. A few weak, long edges remain between groups – two dimensions are not enough to keep every neighbour close.

### The classic comparison: handwritten digits

Before comparing PCA and UMAP on our chunks, one textbook example where the difference is impossible to miss. scikit-learn ships 1,797 images of handwritten digits of 8 × 8 pixels, so every image is a 64-dimensional vector – a small stand-in for an embedding. The labels 0–9 are only used for colouring.

To compare maps with a number we use **trustworthiness** (Venna & Kaski, 2001), `sklearn.manifold.trustworthiness(original, map, n_neighbors=k)`. For every point it looks at the *k* nearest neighbours *in the map* and penalises each one that was not among the *k* nearest neighbours *in the original space* – the more, the farther down the original ranking it was. 1.0 means that every neighbour you see in the picture is a real neighbour; below about 0.9 the picture puts many unrelated points side by side.

In [ ]:
digits, digit_labels = load_digits(return_X_y=True)
digits_pca = PCA(2, random_state=SEED).fit_transform(digits)
digits_umap = umap.UMAP(random_state=SEED, n_jobs=1).fit_transform(digits)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw={"width_ratios": [0.8, 1, 1]})
mosaic = np.zeros((4 * 9, 5 * 9))
for position in range(20):
    row, column = divmod(position, 5)
    example = digits[np.where(digit_labels == position % 10)[0][position // 10]]
    mosaic[row * 9:row * 9 + 8, column * 9:column * 9 + 8] = example.reshape(8, 8)
axes[0].imshow(mosaic, cmap="gray_r")
axes[0].axis("off")
axes[0].set_title(f"{len(digits):,} digits of 8 x 8 pixels = 64-d vectors")
for ax, points, name in [(axes[1], digits_pca, "PCA"), (axes[2], digits_umap, "UMAP")]:
    ax.scatter(points[:, 0], points[:, 1], c=digit_labels, cmap="tab10", s=4)
    for digit in range(10):
        ax.text(*np.median(points[digit_labels == digit], axis=0), str(digit),
                fontsize=13, weight="bold", ha="center", va="center")
    ax.set_title(f"{name}: trustworthiness {trustworthiness(digits, points, n_neighbors=10):.3f}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

PCA's two directions of largest variance mix most digits into one cloud; UMAP draws ten separate islands, one per digit, without ever seeing the labels. The variation in handwriting is spread over many directions, but each digit forms its own connected region of similar images – exactly the kind of structure a neighbour graph captures and a linear projection cannot.

### Our chunks: PCA and UMAP side by side

The same comparison on our corpus. The grey lines connect every chunk to its 5 nearest neighbours in the original 768-dimensional space (cosine distance; adding infinity on the diagonal keeps a chunk from being its own neighbour). A good map keeps those lines short; a long line is a pair of neighbours that the projection tore apart. The titles give the trustworthiness for the same 5 neighbours.

In [ ]:
nearest = np.argsort(distances + np.diag(np.full(len(vectors), np.inf)), axis=1)[:, :5]

fig, axes = plt.subplots(1, 2, figsize=(15, 6.8))
for ax, points, name in [(axes[0], pca_2d, "PCA"), (axes[1], umap_2d, "UMAP")]:
    for i in range(len(points)):
        for j in nearest[i]:
            ax.plot(*zip(points[i], points[j]), color="gray", lw=0.5, alpha=0.6, zorder=0)
    score = trustworthiness(vectors, points, n_neighbors=5, metric="cosine")
    plot_groups(ax, points, f"{name}: trustworthiness {score:.3f}")
legend_below(fig, axes[0])
plt.show()

In 2-D, UMAP keeps the neighbourhoods markedly better (trustworthiness about 0.95 against 0.88). Its groups are compact and most grey lines are short, while PCA's map has many long lines, especially among the overlapping notes of the financial statements. Some of UMAP's groups even split into sub-groups – structure the PCA picture cannot show at all.

## UMAP's knobs

### `n_neighbors`: local versus global

How many neighbours each point considers in step 1. With few neighbours only the closest relationships count, and the map breaks up into many small islands. With many neighbours every point sees a large part of the data: the map shows the coarse arrangement of the groups but blurs their internal structure.

This is the knob the paper turns on purpose. RAPTOR's code first reduces all nodes with `n_neighbors = int(sqrt(n - 1))` – 9 for our 88 chunks, 5 for notebook 5's 30 leaves – and clusters them into *global* clusters. It then runs UMAP again inside every global cluster that has more than 11 members, with `n_neighbors = 10`, to find *local* clusters.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, n_neighbors in zip(axes, [2, 5, 15, 50]):
    points = umap.UMAP(n_neighbors=n_neighbors, metric="cosine", random_state=SEED, n_jobs=1).fit_transform(vectors)
    plot_groups(ax, points, f"n_neighbors = {n_neighbors}")
legend_below(fig, axes[0])
plt.show()

With 2 neighbours the map shatters into pairs and triples; with 5 it forms tight, widely separated clumps; with 15 the groups are visible and connected; with 50 – more than half of the corpus – it turns into one cloud in which only the broad arrangement survives.

### `min_dist`: how tightly points may be packed

The target similarity of step 4 is shaped by `min_dist`, the distance in the layout up to which points count as touching. The left panel plots the curve $1/(1+a\,x^{2b})$ for three values, with *a* and *b* computed by UMAP's own `find_ab_params(spread=1.0, min_dist)`. With `min_dist = 0` neighbours can sit on top of each other and groups become dense clumps; with a large value points spread out evenly, which is easier to read but blurs the edges of the groups. RAPTOR uses UMAP's default, 0.1. For clustering, compact clumps are convenient: they are exactly what a Gaussian mixture separates easily.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
distance_grid = np.linspace(0, 4, 200)
for min_dist, color in [(0.0, "tab:green"), (0.1, "tab:blue"), (0.8, "tab:orange")]:
    a, b = find_ab_params(1.0, min_dist)
    axes[0].plot(distance_grid, 1 / (1 + a * distance_grid ** (2 * b)), color=color, label=f"min_dist = {min_dist}")
axes[0].set_xlabel("distance in the layout")
axes[0].set_title("similarity the layout aims for")
axes[0].legend()
for ax, min_dist in zip(axes[1:], [0.0, 0.1, 0.8]):
    points = umap.UMAP(n_neighbors=10, min_dist=min_dist, metric="cosine",
                       random_state=SEED, n_jobs=1).fit_transform(vectors)
    plot_groups(ax, points, f"min_dist = {min_dist}")
legend_below(fig, axes[1])
plt.show()

### `random_state`: the same neighbourhoods, a different drawing

Step 4 samples random pairs to push apart, so different seeds produce different pictures.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, seed in zip(axes, [0, 1, 2]):
    points = umap.UMAP(n_neighbors=10, metric="cosine", random_state=seed, n_jobs=1).fit_transform(vectors)
    plot_groups(ax, points, f"random_state = {seed}")
legend_below(fig, axes[0])
plt.show()

The islands move, rotate and mirror, but the same chunks keep ending up together. For RAPTOR this means the tree can change from run to run unless the seed is fixed – the paper's code fixes the seed of the Gaussian mixtures but not of UMAP. PCA has no such issue.

## Which method keeps the neighbourhoods at the dimension RAPTOR clusters in?

All maps so far were 2-D, because that is what we can look at. But RAPTOR does not cluster in two dimensions; it reduces to 10 (notebook 5: 8). The comparison that matters is therefore trustworthiness *at the clustering dimension*. We measure it for 2 to 20 output dimensions, both for our 88 chunks (with `n_neighbors = 10`) and for notebook 5's 30 leaves (`load_statements(LEAF_CHUNK_SIZE)`, with `n_neighbors = 5`, the paper's √(n − 1) rule).

In [ ]:
leaf_texts = [text for text, _ in load_statements(LEAF_CHUNK_SIZE)]
leaf_vectors = np.array(embeddings.embed_documents(leaf_texts))
output_dims = [2, 3, 5, 8, 10, 15, 20]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
for ax, data, n_neighbors, name in [(axes[0], vectors, 10, f"our {len(vectors)} chunks"),
                                    (axes[1], leaf_vectors, int((len(leaf_vectors) - 1) ** 0.5),
                                     f"notebook 5's {len(leaf_vectors)} leaves")]:
    pca_scores, umap_scores = [], []
    for dim in output_dims:
        pca_points = PCA(dim, random_state=SEED).fit_transform(data)
        umap_points = umap.UMAP(n_neighbors=n_neighbors, n_components=dim, metric="cosine",
                                random_state=SEED, n_jobs=1).fit_transform(data)
        pca_scores.append(trustworthiness(data, pca_points, n_neighbors=5, metric="cosine"))
        umap_scores.append(trustworthiness(data, umap_points, n_neighbors=5, metric="cosine"))
    ax.plot(output_dims, pca_scores, "o-", label="PCA")
    ax.plot(output_dims, umap_scores, "s-", label=f"UMAP, n_neighbors = {n_neighbors}")
    ax.axvline(10, color="gray", ls=":")
    ax.set_xlabel("output dimensions")
    ax.set_title(name)
    ax.legend()
    print(f"{name}: at 10 dimensions PCA {pca_scores[4]:.3f}, UMAP {umap_scores[4]:.3f}")
axes[0].set_ylabel("trustworthiness (5 neighbours)")
plt.show()

The result turns the 2-D impression around:

- In 2 dimensions UMAP wins clearly on both datasets: a linear projection cannot fold the structure of 768 dimensions into a plane.
- PCA improves with every dimension it keeps; UMAP is close to its ceiling already at 2 dimensions and gains little from more. On the 88 chunks PCA catches up between 8 and 10 dimensions (at 8 UMAP is still slightly ahead) and is in front from 10 on.
- On notebook 5's 30 leaves PCA is ahead from 3 dimensions on, and at the 8 and 10 dimensions used for clustering the gap is large – about 0.95–0.97 against 0.83–0.84. With only five neighbours per point, UMAP's layout is dominated by its tendency to form tight, separate clumps, and some true neighbours end up in different clumps.

## What this means for RAPTOR

- **Reducing is necessary**, mainly because a Gaussian needs *d* + *d*(*d* + 1)/2 numbers per cluster. The embeddings' low intrinsic dimension is what makes the reduction cheap in information.
- **UMAP is the better map.** In 2 or 3 dimensions it keeps neighbourhoods far better than PCA, which is why it is the standard tool for *looking* at embeddings.
- **For clustering in 8–10 dimensions, PCA is fully competitive.** On the 30 leaves that notebook 5 clusters it keeps the neighbourhoods far better than UMAP, and on the larger corpus it draws level by 10 dimensions. Add that it is deterministic and has no knobs, and it is a sound choice for notebook 5. On a large corpus with complex, curved structure the balance can tip; the paper's choice of UMAP is aimed at such corpora.
- **UMAP's output distorts distances and densities by design** (local scaling, `min_dist`), so the gaps between its clumps are not measurements. A mixture fitted to UMAP output clusters UMAP's clumps – often useful, but it means that `n_neighbors` and `min_dist` co-decide the clusters.

Next, notebook 5b fits Gaussian mixtures to points like these.

## Things to try

1. Set `CHUNK_SIZE = 1200` (notebook 5's leaves) and re-run. With 30 points, which of the pictures still work?
2. Run UMAP with `metric="euclidean"` instead of cosine. For unit-length vectors both rank neighbours identically, because $\lVert a-b\rVert^2 = 2 - 2\cos(a,b)$, so the map should hardly change. Does it?
3. Measure trustworthiness with 15 neighbours instead of 5. Does PCA's advantage at 10 dimensions hold for a coarser notion of "neighbour"?
4. Embed the chunks with another Ollama embedding model you have (for example `mxbai-embed-large` or `nomic-embed-text-v2-moe`). Does the contrast of the first figure change?
5. Colour the UMAP map by chunk length instead of by group. Is there a length effect hidden in the layout?